<a href="https://colab.research.google.com/github/jonyxdxp/notebooks_meta/blob/main/v4/s2/v4_2_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ── Cell 1: Mount Drive & clone repo ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from getpass import getpass
import subprocess, os

token    = getpass("GitHub token: ")
repo_dir = '/content/notebooks_meta'

if not os.path.exists(repo_dir):
    subprocess.run(
        ['git', 'clone',
         f'https://{token}@github.com/jonyxdxp/notebooks_meta.git',
         repo_dir], check=True)

%cd {repo_dir}
!git pull origin main

Mounted at /content/drive
GitHub token: ··········
/content/notebooks_meta
From https://github.com/jonyxdxp/notebooks_meta
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:

# ── Cell 3: Imports ───────────────────────────────────────────────────────────

import copy
import sys
import typing

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import datasets
import transformers
import tokenizers

sys.path.insert(0, '/content/notebooks_meta/v4')

from s1.cog_arch.encoder import Encoder
from s2.cog_arch.actor import Actor

from s2.losses import SquareLossSeq, VCLoss

In [5]:
# ── Cell 6: Device ────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if device.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [6]:
# ── Cell 5: Config (single place to change hyperparams) ───────────────────────

# Helper class to allow dot notation access to dictionary keys
class _C:
    def __init__(self, d):
        for k, v in d.items():
            setattr(self, k, _C(v) if isinstance(v, dict) else v)

CFG_dict = dict(
    # Model
    hidden_size  = 256,
    num_heads    = 4,       # 256 / 4 = 64 per head
    num_layers   = 4,
    max_seq_len  = 128,
    mlp_ratio    = 4.0,

    # JEPA masking
    num_target_spans   = 4,
    target_span_length = 8,

    # Training
    lr           = 1e-4,
    weight_decay = 0.05,
    n_epochs     = 20,
    ema_decay     = 0.996,
    batch_size   = 64,
    eval_batch   = 128,
    num_workers  = 0,

    # VICReg
    std_coeff = 50.0,   # was 25.0
    cov_coeff = 1.0,

    # Paths
    cache_dir    = '/content/drive/MyDrive/data/cache',
    ckpt_dir     = '/content/notebooks_meta/v4/1/checkpoints',
    raw_data_dir = '/content/drive/MyDrive/data/dailydialog_raw',  # was /content/data/...
    tokenizer_name = 'bert-base-uncased',
)

CFG = _C(CFG_dict) # Create CFG object supporting dot notation

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu' # Define DEVICE as a global variable

os.makedirs(CFG.ckpt_dir, exist_ok=True) # Access via dot notation

In [7]:
tokenizer = transformers.AutoTokenizer.from_pretrained(CFG.tokenizer_name)
# BERT already has [MASK], [PAD], [CLS], [SEP] — nothing to patch.
# For GPT-style tokenizers that lack these tokens, add them here.

VOCAB_SIZE = tokenizer.vocab_size
print(f"Vocab : {VOCAB_SIZE}  |  mask_token_id : {tokenizer.mask_token_id}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab : 30522  |  mask_token_id : 103


In [8]:
# ── Cell S2-1: Dataset — consecutive turn pairs ───────────────────────────────

def get_turn_pair_dataset(
    cache_dir: str,
    tokenizer,
    block_size: int = 128,
    raw_data_dir: str = CFG.raw_data_dir,
) -> datasets.DatasetDict:
    """
    Builds a dataset of (turn_t, turn_{t+1}) pairs from dialogs.
    Each example: two consecutive utterances from alternating speakers.
    """
    _cache_path = os.path.join(cache_dir, f'turn_pairs_bs{block_size}')
    if os.path.exists(_cache_path):
        print(f'Loading from cache: {_cache_path}')
        return datasets.load_from_disk(_cache_path).with_format('torch')

    split_txts = {
        'train':      os.path.join(raw_data_dir, 'train',      'dialogues_train.txt'),
        'validation': os.path.join(raw_data_dir, 'validation', 'dialogues_validation.txt'),
        'test':       os.path.join(raw_data_dir, 'test',       'dialogues_test.txt'),
    }

    def _load_pairs(filepath):
        """Read dialogs and yield (turn_t, turn_{t+1}) consecutive pairs."""
        pairs_a, pairs_b = [], []   # turn_t, turn_{t+1}
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                turns = [t.strip() for t in line.split('__eou__') if t.strip()]
                # Every consecutive pair: (turn_0→turn_1), (turn_1→turn_2), ...
                for i in range(len(turns) - 1):
                    pairs_a.append(turns[i])
                    pairs_b.append(turns[i + 1])
        return pairs_a, pairs_b

    def _tokenize_pair(examples):
        enc_a = tokenizer(
            examples['turn_a'],
            max_length=block_size, padding='max_length',
            truncation=True, return_attention_mask=True,
            return_token_type_ids=False,
        )
        enc_b = tokenizer(
            examples['turn_b'],
            max_length=block_size, padding='max_length',
            truncation=True, return_attention_mask=True,
            return_token_type_ids=False,
        )
        return {
            'input_ids_a':      enc_a['input_ids'],
            'attention_mask_a': enc_a['attention_mask'],
            'input_ids_b':      enc_b['input_ids'],
            'attention_mask_b': enc_b['attention_mask'],
        }

    print('Building turn-pair dataset …')
    tokenized_splits = {}
    for split, txt_path in split_txts.items():
        a, b = _load_pairs(txt_path)
        print(f'  {split:12s}: {len(a):>6,} turn pairs')
        raw_ds = datasets.Dataset.from_dict({'turn_a': a, 'turn_b': b})
        tokenized_splits[split] = raw_ds.map(
            _tokenize_pair, batched=True, num_proc=1,
            remove_columns=['turn_a', 'turn_b'],
            desc=f'Tokenizing {split}',
        )

    dataset_dict = datasets.DatasetDict(tokenized_splits)
    os.makedirs(cache_dir, exist_ok=True)
    dataset_dict.save_to_disk(_cache_path)
    print(f'Saved to cache: {_cache_path}')
    return dataset_dict.with_format('torch')

In [9]:
# ── Cell S2-2: Collator ───────────────────────────────────────────────────────

class TurnPairCollator:
    """Simple collator — no masking needed, just stack pairs."""
    def __call__(self, batch):
        return {
            'input_ids_a':      torch.stack([b['input_ids_a']      for b in batch]),
            'attention_mask_a': torch.stack([b['attention_mask_a'] for b in batch]),
            'input_ids_b':      torch.stack([b['input_ids_b']      for b in batch]),
            'attention_mask_b': torch.stack([b['attention_mask_b'] for b in batch]),
        }

In [11]:
# ── Cell S2-3: Build dataloaders ──────────────────────────────────────────────

pair_dataset = get_turn_pair_dataset(
    cache_dir = CFG.cache_dir,
    tokenizer = tokenizer,
    block_size = CFG.max_seq_len,
)

collator = TurnPairCollator()

train_loader = torch.utils.data.DataLoader(
    pair_dataset['train'],
    batch_size  = CFG.batch_size,
    shuffle     = True,
    num_workers = 0,
    pin_memory  = True,
    collate_fn  = collator,
)
val_loader = torch.utils.data.DataLoader(
    pair_dataset['validation'],
    batch_size  = CFG.eval_batch,
    shuffle     = False,
    num_workers = 0,
    pin_memory  = True,
    collate_fn  = collator,
)

print(f"Train pairs : {len(pair_dataset['train']):,}")
print(f"Train batches : {len(train_loader)}  |  Val batches : {len(val_loader)}")


Building turn-pair dataset …


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/dailydialog_processed/train/dialogues_train.txt'

In [ ]:
# ── Cell S2-4: Load frozen encoder from Stage 1 ───────────────────────────────

S1_CKPT = os.path.join(CFG.ckpt_dir, 'epoch_020.pt')

context_encoder = Encoder(
    vocab_size  = VOCAB_SIZE,
    hidden_size = CFG.hidden_size,
    num_heads   = CFG.num_heads,
    num_layers  = CFG.num_layers,
    max_seq_len = CFG.max_seq_len,
).to(DEVICE)

ckpt = torch.load(S1_CKPT, map_location=DEVICE)
context_encoder.load_state_dict(ckpt['context_encoder'])
context_encoder.eval()
for p in context_encoder.parameters():
    p.requires_grad = False   # fully frozen

print(f'Loaded Stage 1 encoder from {S1_CKPT}')
print(f'Frozen params : {sum(p.numel() for p in context_encoder.parameters()):,}')




# predictor

predictor = Actor(
    hidden_size = CFG.hidden_size,
    num_heads   = CFG.num_heads,
    num_layers  = 2,
).to(DEVICE)

print(f'Predictor params : {sum(p.numel() for p in predictor.parameters()):,}')



In [ ]:
# ── Cell S2-6: Encoder pooling helper ────────────────────────────────────────

def encode_turn(input_ids, attention_mask):
    """
    Encode a turn with the frozen encoder → mean-pool over real tokens → (B, D).
    No masking needed here — pool all non-padding positions.
    """
    with torch.no_grad():
        hidden = context_encoder(input_ids, attention_mask=attention_mask)
        if isinstance(hidden, tuple):
            hidden = hidden[0]                        # (B, L, D)
    # mean pool over non-padding tokens
    mask_f = attention_mask.unsqueeze(-1).float()     # (B, L, 1)
    pooled = (hidden * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
    return pooled

In [ ]:
# ── Cell S2-7: Loss / optimizer / scheduler ───────────────────────────────────

# Pure MSE between predicted and actual next-turn representation.
# Smooth L1 is slightly more robust to outlier turns.
pred_loss_fn = nn.SmoothL1Loss()

optimizer = AdamW(
    predictor.parameters(),    # encoder is frozen — only train predictor
    lr           = CFG.lr,
    weight_decay = CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.n_epochs, eta_min=CFG.lr * 0.1,
)


In [ ]:
# ── Cell S2-8: Train / eval steps ─────────────────────────────────────────────

def forward_step_s2(batch):
    ids_a  = batch['input_ids_a'].to(DEVICE)
    mask_a = batch['attention_mask_a'].to(DEVICE)
    ids_b  = batch['input_ids_b'].to(DEVICE)
    mask_b = batch['attention_mask_b'].to(DEVICE)

    z_a    = encode_turn(ids_a, mask_a)    # (B, D) — current turn,  no grad
    z_b    = encode_turn(ids_b, mask_b)    # (B, D) — next turn,     no grad

    z_pred = predictor(z_a)                # (B, D) — predicted next turn

    loss   = pred_loss_fn(z_pred, z_b)
    return loss


def train_epoch_s2(loader, epoch):
    predictor.train()
    total, n = 0.0, 0
    pbar = tqdm(loader, desc=f'Epoch {epoch:02d}', leave=False)
    for batch in pbar:
        loss = forward_step_s2(batch)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(predictor.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
        n     += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    return total / n


@torch.no_grad()
def eval_epoch_s2(loader):
    predictor.eval()
    total, n = 0.0, 0
    for batch in loader:
        loss = forward_step_s2(batch)
        total += loss.item()
        n     += 1
    return total / n


# ── Cell S2-9: Checkpointing ──────────────────────────────────────────────────

S2_CKPT_DIR = os.path.join(CFG.ckpt_dir, 'stage2')
os.makedirs(S2_CKPT_DIR, exist_ok=True)

def save_checkpoint_s2(epoch, train_loss, val_loss):
    path = os.path.join(S2_CKPT_DIR, f'epoch_{epoch:03d}.pt')
    torch.save({
        'epoch':      epoch,
        'predictor':  predictor.state_dict(),
        'optimizer':  optimizer.state_dict(),
        'scheduler':  scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss':   val_loss,
    }, path)
    print(f'  ✓ saved → {path}')

In [ ]:
# ── Cell S2-10: Training loop ─────────────────────────────────────────────────

history = {'train_loss': [], 'val_loss': [], 'lr': []}

print(f'\n{"="*60}')
print(f'  Text JEPA Stage 2 — {CFG.n_epochs} epochs   device={DEVICE}')
print(f'  Encoder : FROZEN   Predictor : trainable')
print(f'{"="*60}\n')

for epoch in range(1, CFG.n_epochs + 1):

    train_loss = train_epoch_s2(train_loader, epoch)
    val_loss   = eval_epoch_s2(val_loader)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(optimizer.param_groups[0]['lr'])

    print(
        f'Epoch {epoch:02d}/{CFG.n_epochs}  '
        f'train={train_loss:.4f}  val={val_loss:.4f}  '
        f'lr={optimizer.param_groups[0]["lr"]:.2e}'
    )

    if epoch % 5 == 0:
        save_checkpoint_s2(epoch, train_loss, val_loss)

print('\nStage 2 training complete.')
save_checkpoint_s2(CFG.n_epochs, history['train_loss'][-1], history['val_loss'][-1])

In [ ]:
%%bash
cd /content/notebooks_meta
git config user.email "jonyxdxp@gmail.com"   # only needed once per session
git config user.name  "jonyxdxp"          # only needed once per session

git add v4/2/01.ipynb
git commit -m "update stage 2 notebook"
git push origin main